# Experiment 15 — Gate a specialist for significant underforecasts

Layer 1 is the saved Exp 14 AutoGluon rent ensemble. A binary AutoGluon gate predicts whether Layer 1 will underforecast by more than $1,000. A second AutoGluon expert is trained only on those significant positive residuals. The correction is applied only to rows routed by the gate. Gate threshold and residual weight are selected inside the latest OOF period; the outer validation period remains untouched until final evaluation.

**Executed result:** tuning selected gate threshold **0.05**, residual weight **1.35**, and routed **4.57%** of tuning rows, but gate average precision was only **0.0442**. On outer validation, Layer 1 scored **468.272 RMSE** and the gated specialist **468.360** (MAE **340.929**), a **0.088 deterioration**. The specialist could model large residuals after selection, but the gate could not reliably identify those rows in advance.

In [1]:
from pathlib import Path
import json,os,sys
import numpy as np
import pandas as pd
from autogluon.tabular import TabularPredictor
from sklearn.metrics import average_precision_score,mean_absolute_error,mean_squared_error,r2_score
from sklearn.model_selection import TimeSeriesSplit
configured_root=os.environ.get('KDDM_PROJECT_ROOT')
roots=([Path(configured_root)] if configured_root else [])+[Path.cwd(),*Path.cwd().parents]
project_root=next((root for root in roots if (root/'exploration/metul/preprocessing.py').exists()),None)
if project_root is None:raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
sys.path.insert(0,str(project_root/'exploration'/'metul'))
from preprocessing import RentalPreprocessor,load_datasets
TARGET='MONTHLY_RENT';GATE_TARGET='significant_underforecast';RESIDUAL_TARGET='layer1_residual'
SIGNIFICANT_RESIDUAL=float(os.environ.get('SIGNIFICANT_RESIDUAL','1000'))
TIME_LIMIT=int(os.environ.get('EXP15_TIME_LIMIT','300'));RANDOM_STATE=42;N_SPLITS=3
RUN_ID=os.environ.get('EXP15_RUN_ID',pd.Timestamp.now().strftime('%Y%m%d_%H%M%S'))
HYPERPARAMETERS={'GBM':{},'CAT':{}}
source_nb=json.loads((project_root/'exploration'/'metul'/'exp_14_resale_features_lgbm.ipynb').read_text())
source_cells={cell.get('metadata',{}).get('id'):''.join(cell.get('source',[])) for cell in source_nb['cells']}
exec(source_cells['setup'],globals());exec(source_cells['exp9-best-feature-transformer'],globals())
model_root=project_root/'exploration'/'metul'/'ag_exp_14_ag_residual_models'
requested_run=os.environ.get('EXP15_SOURCE_RUN_ID')
if requested_run:source_run=model_root/requested_run
else:
    candidates=sorted(path for path in model_root.iterdir() if (path/'layer1_full').exists() and (path/'layer1_oof_3').exists())
    if not candidates:raise FileNotFoundError('No completed Exp 14 run found.')
    source_run=candidates[-1]
output_root=project_root/'exploration'/'metul'/'ag_exp_15_significant_residual_gate'/RUN_ID
print(f'Loading Layer 1 models from {source_run.name}; saving Exp 15 models under {output_root}')


Loading Layer 1 models from 20261015_144824; saving Exp 15 models under /Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_15_significant_residual_gate/20261015_153922


In [2]:
_,datasets=load_datasets(project_root/'data')
raw=datasets['train'].copy();raw['period']=pd.to_datetime(raw['RENT_APPROVAL_DATE']+'-01').dt.to_period('M')
periods=np.sort(raw.period.unique());outer_cutoff=periods[int(len(periods)*.80)]
raw_fit,raw_valid=raw.loc[raw.period<outer_cutoff].copy(),raw.loc[raw.period>=outer_cutoff].copy()
def best_features(history,future):
    processor=RentalPreprocessor(datasets)
    history_base=processor.fit_transform(history.drop(columns=['period',TARGET],errors='ignore'))
    future_base=processor.transform(future.drop(columns=['period',TARGET],errors='ignore'))
    return make_exp9_best_features(history,future,history_base,future_base)
def prediction_and_disagreement(predictor,frame):
    prediction=predictor.predict(frame).astype(float).to_numpy()
    base_models=[name for name in predictor.model_names() if ('LightGBM' in name or 'CatBoost' in name) and 'WeightedEnsemble' not in name]
    component=np.vstack([predictor.predict(frame,model=name).astype(float).to_numpy() for name in base_models])
    disagreement=component.max(axis=0)-component.min(axis=0) if len(component)>1 else np.zeros(len(frame))
    return prediction,disagreement
fit_periods=np.sort(raw_fit.period.unique());splits=list(TimeSeriesSplit(n_splits=N_SPLITS).split(fit_periods));early_index,later_index=splits[-1]
early=raw_fit.loc[raw_fit.period.isin(fit_periods[early_index])].copy();latest=raw_fit.loc[raw_fit.period.isin(fit_periods[later_index])].copy()
_,latest_x=best_features(early,latest);oof_model=TabularPredictor.load(str(source_run/'layer1_oof_3'))
latest_prediction,latest_disagreement=prediction_and_disagreement(oof_model,latest_x)
latest_meta=latest_x.copy();latest_meta['layer1_prediction']=latest_prediction;latest_meta['layer1_disagreement']=latest_disagreement;latest_meta[RESIDUAL_TARGET]=latest[TARGET].to_numpy()-latest_prediction;latest_meta[GATE_TARGET]=(latest_meta[RESIDUAL_TARGET]>SIGNIFICANT_RESIDUAL).astype(int);latest_meta['period']=latest.period.to_numpy()
fit_x,valid_x=best_features(raw_fit,raw_valid);layer1=TabularPredictor.load(str(source_run/'layer1_full'))
valid_prediction,valid_disagreement=prediction_and_disagreement(layer1,valid_x)
valid_meta=valid_x.copy();valid_meta['layer1_prediction']=valid_prediction;valid_meta['layer1_disagreement']=valid_disagreement
valid_y=raw_valid[TARGET].astype(float).reset_index(drop=True)
print(f'Latest OOF rows={len(latest_meta):,}; significant underforecast rows={latest_meta[GATE_TARGET].sum():,} ({latest_meta[GATE_TARGET].mean():.2%})')


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


In [3]:
meta_features=BEST_FEATURES+['layer1_prediction','layer1_disagreement']
latest_periods=np.sort(latest_meta.period.unique());meta_cutoff=latest_periods[int(len(latest_periods)*.70)]
meta_train=latest_meta.loc[latest_meta.period<meta_cutoff].reset_index(drop=True);meta_tune=latest_meta.loc[latest_meta.period>=meta_cutoff].reset_index(drop=True)
def fit_gate(frame,path):
    return TabularPredictor(label=GATE_TARGET,problem_type='binary',eval_metric='average_precision',path=str(path)).fit(train_data=frame[meta_features+[GATE_TARGET]],hyperparameters=HYPERPARAMETERS,presets=['medium_quality','ignore_text'],time_limit=TIME_LIMIT,verbosity=2)
def fit_expert(frame,path):
    significant=frame.loc[frame[GATE_TARGET].eq(1)]
    return TabularPredictor(label=RESIDUAL_TARGET,problem_type='regression',eval_metric='root_mean_squared_error',path=str(path)).fit(train_data=significant[meta_features+[RESIDUAL_TARGET]],hyperparameters=HYPERPARAMETERS,presets=['medium_quality','ignore_text'],time_limit=TIME_LIMIT,verbosity=2)
def positive_probability(model,frame):
    probability=model.predict_proba(frame,as_multiclass=True)
    if isinstance(probability,pd.DataFrame):return probability[1].astype(float).to_numpy()
    return np.asarray(probability,dtype=float)
tuning_gate=fit_gate(meta_train,output_root/'tuning_gate');tuning_expert=fit_expert(meta_train,output_root/'tuning_expert')
tune_probability=positive_probability(tuning_gate,meta_tune[meta_features]);tune_residual=tuning_expert.predict(meta_tune[meta_features]).astype(float).to_numpy();tune_base=meta_tune.layer1_prediction.to_numpy();tune_actual=tune_base+meta_tune[RESIDUAL_TARGET].to_numpy()
print(f'Gate tuning average precision: {average_precision_score(meta_tune[GATE_TARGET],tune_probability):.4f}')


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.13 GB / 16.00 GB (25.8%)
Disk Space Avail:   167.87 GB / 460.38 GB (36.5%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_15_significant_residual_gate/20261015_153922/tuning_gate"
Train Data Rows:    21071
Train Data Columns: 20
Label Column:       significant_underforecast
Problem Type:       binary
Preprocessing data...
Select

Gate tuning average precision: 0.0442


In [4]:
threshold_grid=np.linspace(.05,.80,31);weight_grid=np.linspace(0,1.5,61);rows=[]
for threshold in threshold_grid:
    routed=tune_probability>=threshold
    for weight in weight_grid:
        correction=np.where(routed,weight*tune_probability*tune_residual,0.0)
        prediction=np.clip(tune_base+correction,a_min=0,a_max=None)
        rows.append({'gate_threshold':threshold,'residual_weight':weight,'RMSE':mean_squared_error(tune_actual,prediction)**.5,'routed_fraction':routed.mean()})
routing_search=pd.DataFrame(rows);best=routing_search.loc[routing_search.RMSE.idxmin()]
best_threshold=float(best.gate_threshold);best_weight=float(best.residual_weight)
display(routing_search.nsmallest(15,'RMSE'));print(f'Selected threshold={best_threshold:.3f}; weight={best_weight:.3f}; tune routed={best.routed_fraction:.2%}')


,gate_threshold,residual_weight,RMSE,routed_fraction
54,0.05,1.350,531.747780,0.045657
53,0.05,1.325,531.747915,0.045657
55,0.05,1.375,531.748183,0.045657
52,0.05,1.300,531.748587,0.045657
56,0.05,1.400,531.749124,0.045657
51,0.05,1.275,531.749798,0.045657
57,0.05,1.425,531.750604,0.045657
50,0.05,1.250,531.751547,0.045657
58,0.05,1.450,531.752621,0.045657
49,0.05,1.225,531.753833,0.045657


Selected threshold=0.050; weight=1.350; tune routed=4.57%


In [5]:
# Refit the gate and specialist on all latest-OOF rows after selecting hyperparameters.
gate=fit_gate(latest_meta,output_root/'final_gate');expert=fit_expert(latest_meta,output_root/'final_expert')
valid_probability=positive_probability(gate,valid_meta[meta_features]);valid_expert_residual=expert.predict(valid_meta[meta_features]).astype(float).to_numpy();routed=valid_probability>=best_threshold
correction=np.where(routed,best_weight*valid_probability*valid_expert_residual,0.0)
gated_prediction=np.clip(valid_prediction+correction,a_min=0,a_max=None)
oracle_route=(valid_y.to_numpy()-valid_prediction)>SIGNIFICANT_RESIDUAL
def metric(name,prediction):return {'model':name,'RMSE':mean_squared_error(valid_y,prediction)**.5,'MAE':mean_absolute_error(valid_y,prediction),'R2':r2_score(valid_y,prediction)}
comparison=pd.DataFrame([metric('Layer 1 AutoGluon',valid_prediction),metric('Significant-residual gated specialist',gated_prediction)])
display(comparison.sort_values('RMSE'))
display(pd.DataFrame({'gate_probability':valid_probability,'routed':routed,'expert_residual':valid_expert_residual,'applied_correction':correction}).describe())
print(f'Outer validation routed: {routed.mean():.2%}; actual significant underforecast rate: {oracle_route.mean():.2%}')


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.17 GB / 16.00 GB (26.1%)
Disk Space Avail:   167.87 GB / 460.38 GB (36.5%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_15_significant_residual_gate/20261015_153922/final_gate"
Train Data Rows:    30051
Train Data Columns: 20
Label Column:       significant_underforecast
Problem Type:       binary
Preprocessing data...
Selecte

,model,RMSE,MAE,R2
0,Layer 1 AutoGluon,468.271759,340.986810,0.491986
1,Significant-residual gated specialist,468.359583,340.929018,0.491796


,gate_probability,expert_residual,applied_correction
count,32012.000000,32012.000000,32012.000000
mean,0.019314,1178.566243,1.272555
std,0.007524,63.089763,14.872967
min,0.014645,1017.143250,0.000000
25%,0.015742,1137.599640,0.000000
50%,0.017306,1167.981384,0.000000
75%,0.020214,1208.199127,0.000000
max,0.243340,2121.964111,659.154675


Outer validation routed: 0.98%; actual significant underforecast rate: 0.67%
